# OpticIQ, fine-tuning a ViT to detect 16 OCT biomarkers

This project started as **ClinIQ** and was later renamed **OpticIQ**. The Drive folder, Hugging Face repo and token secret keep the ClinIQ name.

This notebook fine-tunes a pretrained ViT-B/16 on the OLIVES dataset to detect 16 biomarkers on a single OCT B-scan. It tunes a threshold for each biomarker, tests the result on patients the model never saw, and publishes the best fold model to Hugging Face for the OpticIQ app.

### Data
* OLIVES labeled subset, 9,408 B-scans from 87 patients (47 TREX DME, 40 PRIME), 192 complete volumes of 49 slices
* 16 biomarkers labeled present or absent on every slice
* Prabhushankar et al., NeurIPS 2022, Zenodo 7105232, CC BY 4.0

### Cells
| Cell | What it does | Runtime |
|---|---|---|
| 0 | Install, imports, Drive paths, `NEED_CACHE` check | CPU |
| 1 to 2 | Unzip the labels and load the biomarker table | CPU |
| 4, 6, 7 | Build the image cache from `OLIVES.zip`, first run only (cell 6 is an optional look at a few slices) | CPU |
| 8 | Unzip the cache and link each row to its image | CPU |
| 9 | Labels as 0 or 1, twelve blank vitreous labels set to absent, patient id from trial and patient number | CPU |
| 10 to 11 | Patient split, 18 test patients and five folds stratified by each patient's rarest biomarker, saved to `common/splits.csv` for the whole team | CPU |
| 12 | Image transforms, resize to 224, gray to 3 channels, pretraining mean and std, training augmentations | GPU |
| 13 | `vit_base_patch16_224.augreg_in21k_ft_in1k` with a new 16-output head | GPU |
| 14 | Scoring and training for one fold, AdamW at 3e-5, one-cycle schedule, weighted loss, best epoch kept, checkpoints every epoch | GPU |
| 15 to 16 | Train fold 0, then the other four | GPU |
| 17 | Out-of-fold predictions and one threshold per biomarker | GPU |
| 18 | Final test on the 18 held-out patients | GPU |
| 19 to 20 | Push the best fold model and `thresholds.json` to Hugging Face, then load it back to check | CPU or GPU |

### Runtime
Cells 0 to 11 run on a CPU. Training and testing need a GPU, ideally an A100 (bfloat16, batch 64). On an L4 or T4 use batch 32, and on a T4 use float16. If Colab disconnects, run all again and training resumes from the last checkpoint.

### Secrets
`CLINIQ_HF_TOKEN` in Colab secrets, with notebook access turned on, for downloading the pretrained model and publishing ours.

### Results
* Best validation macro F1 per fold 0.494, 0.481, 0.474, 0.516 and 0.494
* Held-out test macro F1 0.449 for the fold 3 model, 0.454 for the average of all five
* 0.613 ensemble macro F1 over the 11 biomarkers present in at least three test patients
* Five biomarkers seen in 11 or fewer patients were not learned and are flagged unreliable in the app

### Output
The fold 3 model at `cliniq2027/cliniq-olives-vit-base-224-16bio` on Hugging Face, loaded by the app with `timm.create_model("hf-hub:...")`.

In [ ]:
# CELL 0: Setup, runs every session
!pip -q install timm safetensors huggingface_hub scikit-learn
import os, io, re, json, time, math, zipfile, shutil, random
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm
from PIL import Image
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.metrics import f1_score, roc_auc_score, balanced_accuracy_score
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/ClinIQ"          # folder with the two Zenodo zips
IMAGES_ZIP = f"{DRIVE_DIR}/OLIVES.zip"
LABELS_ZIP = f"{DRIVE_DIR}/OLIVES_Dataset_Labels.zip"
RUN_DIR = f"{DRIVE_DIR}/runs/vijay/vit_base_224"          # checkpoints, splits and results live here
CACHE_ZIP = f"{DRIVE_DIR}/olives_labeled_224.zip"   # resized labeled slices, built once
LOCAL = "/content/olives"                           # fast local disk for training
SPLIT_CSV = f"{DRIVE_DIR}/common/splits.csv"            # shared by the whole team
os.makedirs(os.path.dirname(SPLIT_CSV), exist_ok=True)
os.makedirs(RUN_DIR, exist_ok=True); os.makedirs(LOCAL, exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "no GPU, set a GPU runtime")
for p in [IMAGES_ZIP, LABELS_ZIP]:
    print(os.path.exists(p), p)

# True only until the image cache exists on Drive
NEED_CACHE = not os.path.exists(CACHE_ZIP)
print("cache missing, cells 4 to 7 will build it" if NEED_CACHE else "cache found, cells 4 to 7 will be skipped")

In [ ]:
# CELL 1: unzip the labels once, skip if it was already done this session
LABEL_DIR = f"{LOCAL}/labels"
if not os.path.isdir(LABEL_DIR):
    zipfile.ZipFile(LABELS_ZIP).extractall(LABEL_DIR)

# list every file that came out of the zip, with its size
for root, _, files in os.walk(LABEL_DIR):
    for f in files:
        path = os.path.join(root, f)
        print(f"{os.path.getsize(path)/1e6:8.2f} MB  {path}")

In [ ]:
# CELL 2: Load CSV, copy the path of the biomarker CSV from the list printed above
print(f"{LABEL_DIR}")
csv_path = f"{LABEL_DIR}/OLIVES_Dataset_Labels/full_labels/Biomarker_Clinical_Data_Images.csv"

df = pd.read_csv(csv_path)
print(df.shape)                 # rows and columns
print(df.columns.tolist())      # column names
df.head(3)                      # first 3 rows

print(df.columns[0])
print(df.iloc[:3, 0].tolist())
print(df.iloc[-3:, 0].tolist())

df.head(3)

PATH_COL = df.columns[0]
LABELS = list(df.columns[2:18])

**Whats inside OLIVES.zip?**


In [ ]:
# CELL 4: one time only, copy the trial zips, extract the 9,408 labeled slices and save the cache to Drive
if NEED_CACHE:
    # copy the two trial zips out of OLIVES.zip to local disk, 10 to 30 minutes
    RAW = f"{LOCAL}/raw"
    os.makedirs(RAW, exist_ok=True)
    zf = zipfile.ZipFile(IMAGES_ZIP)
    inner = {}
    for name in ["TREX_DME", "Prime_FULL"]:
        local = f"{RAW}/{name}.zip"
        if not os.path.exists(local):
            print("copying", name, "this takes a while")
            zf.extract(f"OLIVES/{name}.zip", RAW)
            shutil.move(f"{RAW}/OLIVES/{name}.zip", local)
        inner[name] = zipfile.ZipFile(local)

    # read each labeled slice and save it as a numbered png
    IMG_DIR = f"{LOCAL}/img_full"
    os.makedirs(IMG_DIR, exist_ok=True)

    def read_slice(path):
        rel = path.lstrip("/")
        z = inner["TREX_DME"] if rel.startswith("TREX") else inner["Prime_FULL"]
        try:
            return z.read(rel)
        except KeyError:                         # some PRIME slices are png, not tif
            return z.read(rel.replace(".tif", ".png"))

    for i, path in enumerate(tqdm(df[PATH_COL])):
        Image.open(io.BytesIO(read_slice(path))).convert("L").save(f"{IMG_DIR}/{i:05d}.png")

    # zip on local disk first, then copy, so a half written file never lands on Drive
    shutil.make_archive(f"{LOCAL}/cache", "zip", IMG_DIR)
    shutil.copy(f"{LOCAL}/cache.zip", CACHE_ZIP)
    NEED_CACHE = False
    print(len(os.listdir(IMG_DIR)), "images saved, cache written to", CACHE_ZIP)
else:
    print("skipped, cache already on Drive")

**Look at few images from the zip**

In [ ]:
# CELL 6: Not needed only for diagnostics to dump images
if NEED_CACHE:
    N = 6                                         # how many images to dump
    DUMP_DIR = f"{DRIVE_DIR}/sample_images"
    os.makedirs(DUMP_DIR, exist_ok=True)

    PATH_COL = df.columns[0]
    LABELS = list(df.columns[2:18])               # the 16 biomarker columns, Atrophy through SHRM
    trial_zip = {"TREX DME": inner["TREX_DME"], "Prime_FULL": inner["Prime_FULL"]}

    def read_slice(p):
        rel = p.lstrip("/")
        z = trial_zip[rel.split("/")[0]]
        names_in_zip = set(z.namelist()) if not hasattr(z, "_names") else z._names
        z._names = names_in_zip
        if rel not in names_in_zip:               # some PRIME slices are png instead of tif
            rel = next(os.path.splitext(rel)[0] + e for e in (".tif", ".png") if os.path.splitext(rel)[0] + e in names_in_zip)
        return Image.open(io.BytesIO(z.read(rel)))

    # half from each trial
    rows = pd.concat([df[df[PATH_COL].str.startswith("/TREX")].sample(N // 2, random_state=0),
                      df[df[PATH_COL].str.startswith("/Prime")].sample(N - N // 2, random_state=0)])

    fig, axes = plt.subplots(2, (N + 1) // 2, figsize=(5 * ((N + 1) // 2), 10))
    for ax, (i, r) in zip(axes.ravel(), rows.iterrows()):
        img = read_slice(r[PATH_COL])
        present = [l for l in LABELS if r[l] == 1]
        name = f"{i:05d}_" + r[PATH_COL].strip("/").replace("/", "_").replace(" ", "")
        img.save(f"{DUMP_DIR}/{os.path.splitext(name)[0]}.png")
        ax.imshow(img, cmap="gray")
        ax.set_title(r[PATH_COL] + "\n" + ("\n".join(present) or "no biomarkers"), fontsize=8)
        ax.axis("off")
    plt.tight_layout(); plt.show()

    rows[[PATH_COL] + LABELS].to_csv(f"{DUMP_DIR}/labels.csv", index=False)
    print("saved", len(rows), "images and labels.csv to", DUMP_DIR)
else:
    print("skipped, cache already on Drive")


In [ ]:
# CELL 7: pull out the labeled slices at full size and save the cache to Drive
if NEED_CACHE:
    IMG_DIR = f"{LOCAL}/img_full"
    os.makedirs(IMG_DIR, exist_ok=True)

    def read_slice(path):
        rel = path.lstrip("/")                   # CSV paths start with a slash
        z = inner["TREX_DME"] if rel.startswith("TREX") else inner["Prime_FULL"]
        try:
            return z.read(rel)
        except KeyError:                         # some PRIME slices are png, not tif
            return z.read(rel.replace(".tif", ".png"))

    for i, path in enumerate(tqdm(df[PATH_COL])):
        Image.open(io.BytesIO(read_slice(path))).convert("L").save(f"{IMG_DIR}/{i:05d}.png")

    # zip on local disk first, then copy, so a half written file never lands on Drive
    shutil.make_archive(f"{LOCAL}/cache", "zip", IMG_DIR)
    shutil.copy(f"{LOCAL}/cache.zip", CACHE_ZIP)
    NEED_CACHE = False
    print(len(os.listdir(IMG_DIR)), "images saved, cache written to", CACHE_ZIP)
else:
    print("skipped, cache already on Drive")

In [ ]:
# CELL 8: unzip the cache from Drive and link each row to its image
IMG_DIR = f"{LOCAL}/img_full"
if not os.path.isdir(IMG_DIR):
    zipfile.ZipFile(CACHE_ZIP).extractall(IMG_DIR)
df["img"] = [f"{IMG_DIR}/{i:05d}.png" for i in range(len(df))]
print(sum(map(os.path.exists, df["img"])), "of", len(df), "images on local disk")
print(os.path.exists(CACHE_ZIP), os.path.getsize(CACHE_ZIP) / 1e9, "GB")

In [ ]:
# CELL 9: labels as 0 or 1, and a unique patient ID across both trials
# where are the blanks
# 12 slices with a blank vitreous label, we found that some of the
# images have blank labels blank labels become 0, treated as not present
print(df[LABELS].isna().any(axis=1).sum(), "slices had a blank label, filled with 0")
df[LABELS] = df[LABELS].fillna(0).astype(int)

# a unique patient ID across both trials
df["trial"] = df[PATH_COL].str.split("/").str[1]
df["pid"] = df["trial"] + "_" + df["Patient_ID"].astype(str)

print(len(df), "slices from", df["pid"].nunique(), "patients")
print(df.groupby("trial")["pid"].nunique().to_string())
print()
print(df[LABELS].mean().sort_values().round(3).to_string())

In [ ]:
# for each biomarker, how many slices and how many different patients have it
pos = pd.DataFrame({
    "slices": df[LABELS].sum(),
    "patients": [df.loc[df[l] == 1, "pid"].nunique() for l in LABELS],
}).sort_values("patients")
print(pos.to_string())

In [ ]:
# CELL 10: Splitting the dataset based on the patients available for each of the bio marker
# so blind splitting may result in samples not in test set or in training set.

from sklearn.model_selection import StratifiedKFold

# one row per patient, with which biomarkers that patient has on any slice
# note that pid = <trial> + "_" + <patient_id>
patients = df.groupby("pid")[LABELS].max()
print (patients)

# label each patient by the rarest biomarker they have, so rare cases are shared out evenly
n_patients_with = patients.sum()
def rarest(row):
    present = [l for l in LABELS if row[l] == 1]
    if not present:
        return "none"
    r = min(present, key=lambda l: n_patients_with[l])
    return r if n_patients_with[r] < 25 else "common"
patients["group"] = patients.apply(rarest, axis=1)

if os.path.exists(SPLIT_CSV):
    split = pd.read_csv(SPLIT_CSV)
    print("loaded the shared split from Drive")
else:
    # 20% of patients for test, one fold out of five
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    dev_idx, test_idx = next(skf.split(patients, patients["group"]))
    split = pd.DataFrame({"pid": patients.index, "fold": -1})          # -1 means test
    # the other 80% into 5 cross validation folds
    dev = patients.iloc[dev_idx]
    for k, (_, va) in enumerate(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(dev, dev["group"])):
        split.loc[split["pid"].isin(dev.index[va]), "fold"] = k
    split.to_csv(SPLIT_CSV, index=False)
    print("saved a new split to", SPLIT_CSV)

df = df.drop(columns=["fold"], errors="ignore").merge(split, on="pid")

In [ ]:
# CELL 11: check the split
# patients and slices in each part
name = lambda f: "test" if f == -1 else f"fold {f}"
summary = df.groupby("fold").agg(patients=("pid", "nunique"), slices=("pid", "size"))
summary.index = summary.index.map(name)
print(summary.to_string())
print()

# how many patients with each biomarker landed in each part
per_part = df[df[LABELS].eq(1).any(axis=1)].melt(id_vars=["pid", "fold"], value_vars=LABELS)
per_part = per_part[per_part["value"] == 1].groupby(["variable", "fold"])["pid"].nunique().unstack(fill_value=0)
per_part.columns = [name(f) for f in per_part.columns]
print(per_part.loc[n_patients_with.sort_values().index].to_string())

In [ ]:
# CELL 12: Prepare the images, shrink to 224 and do random rotations each time the image is used
# image size, image transforms and the dataset class
import torchvision.transforms as T

MODEL_NAME = "vit_base_patch16_224.augreg_in21k_ft_in1k"
IMG_SIZE = 224

# the mean and std the pretrained model was trained with
cfg = timm.data.resolve_data_config({}, model=timm.create_model(MODEL_NAME, pretrained=False))
MEAN, STD = cfg["mean"], cfg["std"]

# training images get small random changes each time they are used
train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomAffine(degrees=5, translate=(0.03, 0.03), scale=(0.95, 1.05)),
    T.ColorJitter(brightness=0.15, contrast=0.15),
    T.Grayscale(3),                      # the model expects 3 channels, so grayscale is copied 3 times
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

# validation and test images are only resized, never changed
eval_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.Grayscale(3),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

# gives the model one slice and its 16 labels at a time
class OCTSlices(Dataset):
    def __init__(self, rows, tf):
        self.paths = rows["img"].tolist()
        self.labels = rows[LABELS].values.astype("float32")
        self.tf = tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        return self.tf(Image.open(self.paths[i])), torch.from_numpy(self.labels[i])

# quick check on one slice
x, y = OCTSlices(df.head(1), train_tf)[0]
print("image tensor", tuple(x.shape), "labels", y.int().tolist())

In [ ]:
# CELL 13, the model, batch size and data loaders
# training settings and the model
BATCH = 64                  # 64 fits an A100 easily, use 32 on an L4 or T4
WORKERS = 8                 # processes loading images in parallel
AMP_DTYPE = torch.bfloat16  # faster, lower memory number format on A100

# pretrained ViT-Base from Hugging Face, with a new final layer giving 16 biomarker scores
def make_model():
    return timm.create_model(MODEL_NAME, pretrained=True, num_classes=len(LABELS), drop_path_rate=0.1).to(DEVICE)

# check that one batch of slices goes through the model
model = make_model()
images, labels = next(iter(DataLoader(OCTSlices(df.head(BATCH), eval_tf), batch_size=BATCH)))
with torch.no_grad(), torch.autocast("cuda", dtype=AMP_DTYPE):
    scores = model(images.to(DEVICE))

print(torch.cuda.get_device_name(0))
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters")
print("in", tuple(images.shape), "out", tuple(scores.shape))
del model

In [ ]:
# CELL 14, scoring and training for one fold
EPOCHS = 6
LR = 3e-5


def predict(model, rows):
    """probabilities from 0 to 1, one row per slice and one column per biomarker"""
    model.eval()
    loader = DataLoader(OCTSlices(rows, eval_tf), batch_size=BATCH, num_workers=WORKERS)
    probs = []
    with torch.no_grad(), torch.autocast("cuda", dtype=AMP_DTYPE):
        for images, _ in loader:
            probs.append(torch.sigmoid(model(images.to(DEVICE))).float().cpu())
    return torch.cat(probs).numpy()


def biomarker_f1(probs, labels, thresholds=0.5):
    """
    F1 for each of the 16 biomarkers, returned as an array of 16 numbers.

    precision = tp / (tp + fp)
    recall    = tp / (tp + fn)
    F1        = 2 * precision * recall / (precision + recall)

    thresholds can be one number for all biomarkers or 16 numbers, one each.
    Biomarkers with no positive slices get NaN, and np.nanmean skips them when averaging.
    """
    predicted = probs >= np.asarray(thresholds)             # True where the model says present
    present = labels == 1                                    # True where it really is present

    tp = (predicted & present).sum(axis=0)                   # counts per biomarker, column by column
    fp = (predicted & ~present).sum(axis=0)
    fn = (~predicted & present).sum(axis=0)

    precision = tp / np.maximum(tp + fp, 1)                  # max with 1 avoids dividing by zero
    recall = tp / np.maximum(tp + fn, 1)
    f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-9)

    f1[present.sum(axis=0) == 0] = np.nan                    # no positives, no meaningful F1
    return f1


def train_fold(k):
    fold_dir = f"{RUN_DIR}/fold_{k}"
    os.makedirs(fold_dir, exist_ok=True)
    if os.path.exists(f"{fold_dir}/done.json"):
        print(f"fold {k} already finished")
        return json.load(open(f"{fold_dir}/done.json"))

    # this fold is validation, the other four folds are training, test (-1) is never used
    train_rows = df[(df["fold"] >= 0) & (df["fold"] != k)]
    val_rows = df[df["fold"] == k]
    loader = DataLoader(OCTSlices(train_rows, train_tf), batch_size=BATCH, shuffle=True,
                        num_workers=WORKERS, drop_last=True, persistent_workers=True)

    # rare biomarkers get more weight in the loss, capped at 20
    prevalence = train_rows[LABELS].values.mean(axis=0).clip(1e-3, 1)
    pos_weight = torch.tensor((1 - prevalence) / prevalence, dtype=torch.float32).clamp(1, 20).to(DEVICE)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    model = make_model()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05)
    schedule = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=LR, total_steps=EPOCHS * len(loader), pct_start=0.1)
    start_epoch, best_f1, history = 0, -1.0, []

    # resume from the last saved epoch if Colab disconnected
    if os.path.exists(f"{fold_dir}/last.pt"):
        saved = torch.load(f"{fold_dir}/last.pt", map_location=DEVICE, weights_only=False)
        model.load_state_dict(saved["model"])
        optimizer.load_state_dict(saved["optimizer"])
        schedule.load_state_dict(saved["schedule"])
        start_epoch, best_f1, history = saved["epoch"] + 1, saved["best_f1"], saved["history"]
        print(f"fold {k} resuming at epoch {start_epoch}")

    for epoch in range(start_epoch, EPOCHS):
        model.train()
        started, total_loss = time.time(), 0.0
        for images, labels in tqdm(loader, desc=f"fold {k} epoch {epoch}", leave=False):
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                loss = loss_fn(model(images.to(DEVICE)), labels.to(DEVICE))
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            schedule.step()
            total_loss += loss.item()

        # macro F1, the average F1 over the biomarkers present in this fold
        val_f1 = float(np.nanmean(biomarker_f1(predict(model, val_rows), val_rows[LABELS].values)))
        history.append({"epoch": epoch, "loss": total_loss / len(loader), "val_macro_f1": val_f1})
        print(f"fold {k} epoch {epoch}  loss {total_loss / len(loader):.4f}  val macro F1 {val_f1:.4f}  "
              f"{(time.time() - started) / 60:.1f} min")

        # keep the best epoch's weights, and save everything needed to resume
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), f"{fold_dir}/best.pt")
        torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "schedule": schedule.state_dict(),
                    "epoch": epoch, "best_f1": best_f1, "history": history}, f"{fold_dir}/last.pt")

    result = {"fold": k, "best_val_macro_f1": best_f1, "history": history}
    json.dump(result, open(f"{fold_dir}/done.json", "w"), indent=2)
    os.remove(f"{fold_dir}/last.pt")
    del model
    torch.cuda.empty_cache()
    return result

In [ ]:
# CELL 15, train one fold first to check timing and that the score improves
result = train_fold(0)
pd.DataFrame(result["history"])

In [ ]:
# CELL 16, train all 5 folds, fold 0 is skipped since it is already done
fold_results = [train_fold(k) for k in range(5)]
pd.DataFrame([{"fold": r["fold"], "best_val_macro_f1": r["best_val_macro_f1"]} for r in fold_results])

In [ ]:
# CELL 17, out of fold predictions, per biomarker thresholds and cross validation scores

# load the best saved model for one fold
def load_fold_model(k):
    model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=len(LABELS))
    model.load_state_dict(torch.load(f"{RUN_DIR}/fold_{k}/best.pt", map_location=DEVICE))
    return model.to(DEVICE)

# step 1, each fold model predicts its own validation patients, then join the five folds
fold_probs, fold_true, fold_rows = [], [], []
for k in range(5):
    val_rows = df[df["fold"] == k]
    model = load_fold_model(k)
    fold_probs.append(predict(model, val_rows))      # probabilities, sigmoid applied inside predict
    fold_true.append(val_rows[LABELS].values)        # the true labels for the same rows, same order
    fold_rows.append(val_rows)
    del model

# out of fold (oof) - prediction made on the fold that was held out
oof = np.concatenate(fold_probs)                     # all development slices, fold 0 first, then 1 to 4
true = np.concatenate(fold_true)
dev = pd.concat(fold_rows).reset_index(drop=True)    # the matching rows, used for patient counts

# step 2, score all 16 biomarkers at each candidate threshold, then keep each biomarker's best
candidates = np.arange(0.05, 0.96, 0.05)
scores = np.array([biomarker_f1(oof, true, t) for t in candidates])     # one row per threshold
thresholds = candidates[np.argmax(scores, axis=0)]                      # best row for each biomarker

# biomarkers the model did not learn keep the default 0.5, so the app does not over call them
not_learned = ["PED (serous)", "VMT", "DRIL", "Disruption of RPE", "Atrophy / thinning of retinal layers"]
for name in not_learned:
    thresholds[LABELS.index(name)] = 0.5

# step 3, score each biomarker at 0.5 and at its own threshold
f1_at_half = biomarker_f1(oof, true, 0.5)
f1_tuned = biomarker_f1(oof, true, thresholds)

cv_table = pd.DataFrame({
    "biomarker": LABELS,
    "patients": [dev.loc[dev[name] == 1, "pid"].nunique() for name in LABELS],
    "F1 at 0.5": f1_at_half,
    "threshold": thresholds,
    "F1 tuned": f1_tuned,
    "AUROC": [roc_auc_score(true[:, j], oof[:, j]) for j in range(len(LABELS))],
}).sort_values("patients")

print("macro F1 at 0.5 ", round(np.nanmean(f1_at_half), 3))
print("macro F1 tuned  ", round(np.nanmean(f1_tuned), 3))
cv_table.round(3)

In [ ]:
# cell 18, final evaluation on the 18 test patients, which played no part in training or tuning
from sklearn.metrics import average_precision_score

# step 1, the test rows and their true labels
test_rows = df[df["fold"] == -1]
test_true = test_rows[LABELS].values
print(len(test_rows), "test slices from", test_rows["pid"].nunique(), "patients")
print()

# step 2, each of the 5 fold models predicts every test slice
fold_test_probs = []
for k in range(5):
    model = load_fold_model(k)
    probs = predict(model, test_rows)
    fold_test_probs.append(probs)
    del model

# step 3, average the five models' probabilities slice by slice
avg_probs = np.mean(fold_test_probs, axis=0)

# step 4, macro F1 for each model alone, then for the average
for k in range(5):
    f1_per_biomarker = biomarker_f1(fold_test_probs[k], test_true, thresholds)
    macro = np.nanmean(f1_per_biomarker)
    print(f"fold {k} model      macro F1 {macro:.3f}")

avg_f1_per_biomarker = biomarker_f1(avg_probs, test_true, thresholds)
avg_macro = np.nanmean(avg_f1_per_biomarker)
print(f"average of all 5   macro F1 {avg_macro:.3f}")
print()

# step 5, one row per biomarker for the average of the 5 models
rows = []
for j, name in enumerate(LABELS):
    true_j = test_true[:, j]
    probs_j = avg_probs[:, j]

    # how many test patients have this biomarker
    patients_with_it = test_rows.loc[test_rows[name] == 1, "pid"].nunique()

    # AUROC and average precision need at least one positive and one negative slice
    has_both = 0 < true_j.sum() < len(true_j)
    if has_both:
        auroc = roc_auc_score(true_j, probs_j)
        avg_precision = average_precision_score(true_j, probs_j)
    else:
        auroc = np.nan
        avg_precision = np.nan

    rows.append({
        "biomarker": name,
        "test patients": patients_with_it,
        "threshold": thresholds[j],
        "F1": avg_f1_per_biomarker[j],
        "AUROC": auroc,
        "avg precision": avg_precision,
        "enough data": patients_with_it >= 3,
    })

test_table = pd.DataFrame(rows).sort_values("test patients")

# step 6, the headline number, macro F1 over biomarkers with at least 3 test patients
measurable = test_table[test_table["enough data"]]
print(f"macro F1 over the {len(measurable)} biomarkers with at least 3 test patients   {measurable['F1'].mean():.3f}")
test_table.round(3)

In [ ]:
# CELL 19, push the best fold model to Hugging Face as a standard timm model
from google.colab import userdata
from huggingface_hub import login, HfApi

# step 1, log in with the token saved under the key icon
login(token=userdata.get("CLINIQ_HF_TOKEN"))
api = HfApi()
REPO_ID = f"{api.whoami()['name']}/cliniq-olives-vit-base-224-16bio"

# step 2, load the fold with the best validation F1
best_fold = int(np.argmax([r["best_val_macro_f1"] for r in fold_results]))
model = load_fold_model(best_fold).cpu()
print("uploading fold", best_fold)

# step 3, push the weights and config, label_names records which output is which biomarker
timm.models.push_to_hf_hub(
    model,
    REPO_ID,
    private=True,
    model_config={"label_names": LABELS},
    commit_message=f"ViT-Base fold {best_fold}, 16 OLIVES biomarkers",
)

# step 4, the per biomarker thresholds as a separate small file, timm has no place for them
thresholds_file = f"{LOCAL}/thresholds.json"
json.dump({name: float(t) for name, t in zip(LABELS, thresholds)}, open(thresholds_file, "w"), indent=2)
api.upload_file(path_or_fileobj=thresholds_file, path_in_repo="thresholds.json", repo_id=REPO_ID)

print("uploaded to https://huggingface.co/" + REPO_ID)
print(api.list_repo_files(REPO_ID))

In [ ]:
# CELL 20, load the model back from Hugging Face and compare with the local copy
from huggingface_hub import hf_hub_download

# step 1, load exactly the way anyone else would
hub_model = timm.create_model("hf-hub:" + REPO_ID, pretrained=True).to(DEVICE)
print("label names", hub_model.pretrained_cfg.get("label_names"))

# step 2, run the same 64 test slices through both and compare
sample = test_rows.head(64)
probs_hub = predict(hub_model, sample)
probs_local = fold_test_probs[best_fold][:64]
print("largest difference", float(np.abs(probs_hub - probs_local).max()))

# step 3, read the thresholds back from the repo
hub_thresholds = json.load(open(hf_hub_download(REPO_ID, "thresholds.json")))
print(hub_thresholds)